# Previsão de jogos da Liga Portugal: modelos

**Fase 3: referências.** Antes de treinar qualquer modelo, medem-se as regras simples
que ele tem de bater, na época de validação (2024/25).

Medidas (quanto mais baixo melhor, exceto o acerto):
- **Acerto:** o resultado mais provável aconteceu?
- **Log loss:** média de -log(probabilidade dada ao resultado real). Adivinhar ao acaso dá 1,10.
- **Brier:** soma dos quadrados entre as probabilidades e o que aconteceu (1 no real, 0 nos outros).

In [1]:
import sqlite3
from pathlib import Path
from contextlib import closing

import numpy as np
import pandas as pd

PASTA_PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
with closing(sqlite3.connect(PASTA_PROJETO / "dados" / "futebol.db")) as ligacao:
    variaveis = pd.read_sql("SELECT * FROM variaveis ORDER BY Data, JogoID", ligacao, parse_dates=["Data"])

treino = variaveis[variaveis["Uso"] == "Treino"]
validacao = variaveis[variaveis["Uso"] == "Validação"]
print(f"Treino: {len(treino)} jogos | Validacao: {len(validacao)} jogos")

Treino: 918 jogos | Validacao: 1224 jogos


In [2]:
RESULTADOS = ["H", "D", "A"]

def avaliar(nome,probabilidades,resultado_real):
    probabilidades = np.asarray(probabilidades, dtype=float)
    real = pd.Series(resultado_real).to_numpy()

    previsto = np.array(RESULTADOS)[probabilidades.argmax(axis=1)]
    acerto = (previsto == real).mean()

    real_01 = np.column_stack([(real == r).astype(float) for r in RESULTADOS])
    brier = ((probabilidades - real_01) ** 2).sum(axis=1).mean()
    
    prob_do_real = (probabilidades * real_01).sum(axis=1)
    perda = -np.log(np.clip(prob_do_real, 1e-15, 1)).mean()

    return {"Modelo": nome, "Acerto": round(acerto, 3), "LogLoss": round(perda, 3), "Brier": round(brier, 3)}


In [3]:
frequencias = treino["Resultado"].value_counts(normalize=True).reindex(RESULTADOS)
print(frequencias.round(3))
prob_frequencias = np.tile(frequencias.to_numpy(), (len(validacao), 1))
r1 = avaliar("Frequências do treino", prob_frequencias, validacao["Resultado"])

Resultado
H    0.431
D    0.237
A    0.331
Name: proportion, dtype: float64


In [4]:
# Os limites dos escalões vêm só do treino
_, limites = pd.qcut(treino["DifElo"], 5, retbins=True)
limites[0], limites[-1] = -np.inf, np.inf
escalao_treino = pd.cut(treino["DifElo"], limites)
tabela_elo = pd.crosstab(escalao_treino, treino["Resultado"], normalize="index")[RESULTADOS]
display(tabela_elo.round(2))

escalao_validacao = pd.cut(validacao["DifElo"], limites)
prob_elo = tabela_elo.loc[escalao_validacao].to_numpy()
r2 = avaliar("Elo por escalões", prob_elo, validacao["Resultado"])

Resultado,H,D,A
DifElo,,,
"(-inf, -100.567]",0.17,0.21,0.62
"(-100.567, -23.816]",0.34,0.27,0.39
"(-23.816, 23.655]",0.40,0.32,0.29
"(23.655, 96.19]",0.46,0.26,0.27
"(96.19, inf]",0.78,0.14,0.09


In [5]:
prob_odds = validacao[["ProbOddsCasa", "ProbOddsEmpate", "ProbOddsFora"]].to_numpy()
r3 = avaliar("Odds (Bet365)", prob_odds, validacao["Resultado"])

In [6]:
referencias = pd.DataFrame([r1, r2, r3])
referencias

,Modelo,Acerto,LogLoss,Brier
0,Frequências do treino,0.438,1.070,0.647
1,Elo por escalões,0.542,0.949,0.560
2,Odds (Bet365),0.572,0.913,0.536


In [7]:
import sys
sys.path.insert(0, str(PASTA_PROJETO / "etl"))
from func.variaveis import colunas_do_modelo

colunas = colunas_do_modelo(variaveis)
X_treino, y_treino = treino[colunas], treino["Resultado"]
X_validacao, y_validacao = validacao[colunas], validacao["Resultado"]
print(f"{len(colunas)} variaveis | treino: {len(X_treino.shape)} | validacao: {len(X_validacao.shape)}")

37 variaveis | treino: 2 | validacao: 2


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modelo.fit(X_treino, y_treino)

print("Treinado. Classes:", modelo.classes_)

Treinado. Classes: ['A' 'D' 'H']


In [9]:
prob_modelo = pd.DataFrame(modelo.predict_proba(X_validacao), columns=modelo.classes_)
prob_modelo = prob_modelo[RESULTADOS]
prob_modelo.head()

,H,D,A
0,0.884201,0.061989,0.053811
1,0.347348,0.290480,0.362172
2,0.161149,0.233301,0.605550
3,0.069977,0.259751,0.670272
4,0.445392,0.255252,0.299356


In [10]:
r4 = avaliar("Modelo de regressão logística", prob_modelo, y_validacao)
pd.DataFrame([r1, r2, r4, r3])

,Modelo,Acerto,LogLoss,Brier
0,Frequências do treino,0.438,1.070,0.647
1,Elo por escalões,0.542,0.949,0.560
2,Modelo de regressão logística,0.545,0.956,0.561
3,Odds (Bet365),0.572,0.913,0.536


In [11]:
pesos = pd.DataFrame(modelo[-1].coef_.T, index=colunas, columns=modelo.classes_)[RESULTADOS]
pesos.round(2).sort_values(by="H", ascending=False)

,H,D,A
PiDifEsperada,0.50,-0.25,-0.25
EquilibrioElo,0.28,-0.19,-0.09
GapFalhadosCasa,0.19,-0.15,-0.03
PiCasa,0.13,-0.47,0.35
ForaGolosSofridosUlt5,0.12,0.02,-0.13
GapFalhadosDif,0.12,-0.13,0.01
ForaPontosUlt5,0.11,0.01,-0.12
CasaGolosSofridosUlt5,0.09,-0.09,0.01
ForaRematesBalizaUlt5,0.09,-0.07,-0.02
GapCantosCasa,0.08,-0.05,-0.03
